# Lenguajes y Autómatas II (SCD-1016)
## Sesión 12 · Atributos de Símbolos y Cálculo de Memoria

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cuántos bytes ocupa cada variable y por qué el compilador no puede sumar tamaños sin más?
>
> **Objetivo de la sesión:** Calcular el ancho de cada tipo y el desplazamiento que le corresponde a la siguiente variable dentro del marco de pila, respetando la alineación natural de la arquitectura.

**Entregable:** tus funciones ancho_de_tipo() y asignar_offset() en artesano_sesion12_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion12.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un registro de símbolo con su nombre y su tipo, más el cursor donde terminó la variable anterior; asignar_offset() redondea el cursor, escribe el offset en el registro y devuelve dónde empieza el siguiente. ancho_de_tipo() devuelve el tamaño en bytes o -1 si el tipo no existe.

Al final de la sesión tendrás: **Un calculador que llena el registro de un símbolo con su offset alineado y devuelve el cursor para el siguiente, con pruebas de char, short, int y double.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 12](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion12.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion12.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion12.h"

int ancho_de_tipo(const char *tipo);
int asignar_offset(Simbolo *simbolo, int cursor);
```

- Los tipos válidos son char, short, int, float, double y puntero; cualquier otro devuelve -1.
- El cursor nunca retrocede y el offset guardado siempre es múltiplo del ancho del tipo.
- Un registro NULL, un cursor negativo o un tipo desconocido se rechazan con -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion12_c.c` junto con `artesano_sesion12_c.c`
> (o `ia_sesion12_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `ancho_de_tipo("int")` | 4 |
| 2 | `ancho_de_tipo("double")` | 8 |
| 3 | `int en cursor 0` | 4 |
| 4 | `char e int: offset de b` | 4 |
| 5 | `double desde cursor 4` | 8 |
| 6 | `registro NULL` | -1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 12 - Registro de simbolo y offsets alineados ===
Caso                               esperado:              obtenido:              resultado
[ 1] ancho_de_tipo("int")               esperado: 4            obtenido: 4            OK
[ 2] ancho_de_tipo("double")            esperado: 8            obtenido: 8            OK
[ 3] int en cursor 0                    esperado: 4            obtenido: 4            OK
[ 4] char e int: offset de b            esperado: 4            obtenido: 4            OK
[ 5] double desde cursor 4              esperado: 8            obtenido: 8            OK
[ 6] registro NULL                      esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion12.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] ancho_de_tipo() respeta la tabla char=1, short=2, int=4, float=4, double=8, puntero=8.
- [ ] asignar_offset() guarda en el registro el offset ya alineado y devuelve el cursor del siguiente.
- [ ] El registro NULL devuelve -1 y no escribe memoria inválida.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion12_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion12.h
/* api_sesion12.h - contrato de la Sesion 12
 * Registro de símbolo y offsets alineados
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION12_H
#define API_SESION12_H

typedef struct Simbolo {
    const char *nombre;
    const char *tipo;
    int offset;
} Simbolo;

int ancho_de_tipo(const char *tipo);
int asignar_offset(Simbolo *simbolo, int cursor);

#endif /* API_SESION12_H */


In [ ]:
%%writefile artesano_sesion12_c.c
/* artesano_sesion12_c.c - Fase 1: El Artesano (sin IA)
 * Registro de símbolo y offsets alineados.
 *
 * Implementa lo que declara api_sesion12.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion12.h"
#include <stdio.h>
#include <string.h>

int ancho_de_tipo(const char *tipo) {
    (void)tipo; /* TODO: char=1, short=2, int=4, double=8 */
    return -1;
}

int asignar_offset(Simbolo *simbolo, int cursor) {
    (void)simbolo; (void)cursor; /* TODO: alinea el cursor y guarda el offset */
    return -1;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion12_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion12_c.c
/* ia_sesion12_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion12.h. No puede traer main().
 */
#include "api_sesion12.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion12_c.c
/* harness_sesion12_c.c - auditoria automatica de la Sesion 12
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion12_c.c artesano_sesion12_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion12.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 12 - Registro de simbolo y offsets alineados ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int r = ancho_de_tipo("int"); verificar_int("ancho_de_tipo(\"int\")", (long)(4), (long)(r)); }
    { int r = ancho_de_tipo("double"); verificar_int("ancho_de_tipo(\"double\")", (long)(8), (long)(r)); }
    { Simbolo s = {"x", "int", -1}; int r = asignar_offset(&s, 0); verificar_int("int en cursor 0", (long)(4), (long)(r)); }
    { Simbolo a = {"a", "char", -1}; Simbolo b = {"b", "int", -1}; int cursor = asignar_offset(&a, 0); asignar_offset(&b, cursor); int r = b.offset; verificar_int("char e int: offset de b", (long)(4), (long)(r)); }
    { Simbolo d = {"d", "double", -1}; asignar_offset(&d, 4); int r = d.offset; verificar_int("double desde cursor 4", (long)(8), (long)(r)); }
    { int r = asignar_offset(NULL, 0); verificar_int("registro NULL", (long)(-1), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion12.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion12_c.c artesano_sesion12_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion12_c.c ia_sesion12_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion12.md` a `auditorias/auditoria_sesion12.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
